# Задача 3. Машинный перевод (EN → RU) и BLEU

3. Решить задачу машинного перевода, выбрав свой язык:

* **Формируем датасет** с исходного языка на целевой (код прописать в классе).
* **Строим архитектуру** нейронной сети.
* **Обучаем**.
* **Проверяем качество** с помощью метрики BLEU.

**2 балла** за правильно выполненное задание.

In [1]:
# Блок 0. Импорты

import math
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
# Блок 1. Небольшой корпус EN-RU

pairs = [
    ("i like cats", "мне нравятся кошки"),
    ("i like dogs", "мне нравятся собаки"),
    ("you like cats", "тебе нравятся кошки"),
    ("you like dogs", "тебе нравятся собаки"),
    ("cats are animals", "кошки это животные"),
    ("dogs are animals", "собаки это животные"),
    ("i have a dog", "у меня есть собака"),
    ("i have a cat", "у меня есть кошка"),
    ("you have a dog", "у тебя есть собака"),
    ("you have a cat", "у тебя есть кошка"),
]

print("Всего пар:", len(pairs))
print("Пример:", pairs[0])

Всего пар: 10
Пример: ('i like cats', 'мне нравятся кошки')


In [3]:
# Блок 2. Токенизация и словари

SRC_PAD = "<pad>"
SRC_SOS = "<sos>"
SRC_EOS = "<eos>"

TGT_PAD = "<pad>"
TGT_SOS = "<sos>"
TGT_EOS = "<eos>"

def tokenize_en(s):
    return s.lower().split()

def tokenize_ru(s):
    return s.lower().split()

def build_vocab(sentences, pad, sos, eos, tokenizer):
    all_tokens = []
    for s in sentences:
        all_tokens.extend(tokenizer(s))
    vocab = [pad, sos, eos] + sorted(list(set(all_tokens)))
    stoi = {}
    itos = {}
    for i, w in enumerate(vocab):
        stoi[w] = i
        itos[i] = w
    return vocab, stoi, itos

src_sentences = [src for src, tgt in pairs]
tgt_sentences = [tgt for src, tgt in pairs]

src_vocab, src_stoi, src_itos = build_vocab(src_sentences, SRC_PAD, SRC_SOS, SRC_EOS, tokenize_en)
tgt_vocab, tgt_stoi, tgt_itos = build_vocab(tgt_sentences, TGT_PAD, TGT_SOS, TGT_EOS, tokenize_ru)

SRC_PAD_IDX = src_stoi[SRC_PAD]
SRC_SOS_IDX = src_stoi[SRC_SOS]
SRC_EOS_IDX = src_stoi[SRC_EOS]

TGT_PAD_IDX = tgt_stoi[TGT_PAD]
TGT_SOS_IDX = tgt_stoi[TGT_SOS]
TGT_EOS_IDX = tgt_stoi[TGT_EOS]

print("SRC vocab:", src_vocab)
print("TGT vocab:", tgt_vocab)

SRC vocab: ['<pad>', '<sos>', '<eos>', 'a', 'animals', 'are', 'cat', 'cats', 'dog', 'dogs', 'have', 'i', 'like', 'you']
TGT vocab: ['<pad>', '<sos>', '<eos>', 'есть', 'животные', 'кошка', 'кошки', 'меня', 'мне', 'нравятся', 'собака', 'собаки', 'тебе', 'тебя', 'у', 'это']


In [4]:
# Блок 3. Кодирование предложений

MAX_SRC_LEN = 12
MAX_TGT_LEN = 12

def encode_src(sentence, max_len=MAX_SRC_LEN):
    tokens = [SRC_SOS] + tokenize_en(sentence) + [SRC_EOS]
    ids = [src_stoi[t] for t in tokens]
    if len(ids) < max_len:
        ids += [SRC_PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return torch.tensor(ids, dtype=torch.long)

def encode_tgt(sentence, max_len=MAX_TGT_LEN):
    tokens = tokenize_ru(sentence)
    inp_tokens = [TGT_SOS] + tokens
    out_tokens = tokens + [TGT_EOS]

    inp_ids = [tgt_stoi[t] for t in inp_tokens]
    out_ids = [tgt_stoi[t] for t in out_tokens]

    if len(inp_ids) < max_len:
        inp_ids += [TGT_PAD_IDX] * (max_len - len(inp_ids))
    else:
        inp_ids = inp_ids[:max_len]

    if len(out_ids) < max_len:
        out_ids += [TGT_PAD_IDX] * (max_len - len(out_ids))
    else:
        out_ids = out_ids[:max_len]

    return torch.tensor(inp_ids, dtype=torch.long), torch.tensor(out_ids, dtype=torch.long)

In [5]:
# Блок 4. Dataset и DataLoader

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_ids = encode_src(src)
        tgt_inp_ids, tgt_out_ids = encode_tgt(tgt)
        return src_ids, tgt_inp_ids, tgt_out_ids

dataset = TranslationDataset(pairs)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

src_batch, tgt_inp_batch, tgt_out_batch = next(iter(dataloader))
print("Batch shapes:", src_batch.shape, tgt_inp_batch.shape, tgt_out_batch.shape)

Batch shapes: torch.Size([4, 12]) torch.Size([4, 12]) torch.Size([4, 12])


In [6]:
# Блок 5. Seq2Seq модель (Encoder-Decoder на GRU)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=SRC_PAD_IDX)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        emb = self.embed(src)
        outputs, h_n = self.rnn(emb)
        return h_n

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=TGT_PAD_IDX)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt_inp, hidden):
        emb = self.embed(tgt_inp)
        outputs, h_n = self.rnn(emb, hidden)
        logits = self.fc(outputs)
        return logits, h_n

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt_inp):
        hidden = self.encoder(src)
        logits, _ = self.decoder(tgt_inp, hidden)
        return logits

enc = Encoder(len(src_vocab), emb_dim=32, hidden_dim=64)
dec = Decoder(len(tgt_vocab), emb_dim=32, hidden_dim=64)
model = Seq2Seq(enc, dec).to(device)
print(model)

Seq2Seq(
  (encoder): Encoder(
    (embed): Embedding(14, 32, padding_idx=0)
    (rnn): GRU(32, 64, batch_first=True)
  )
  (decoder): Decoder(
    (embed): Embedding(16, 32, padding_idx=0)
    (rnn): GRU(32, 64, batch_first=True)
    (fc): Linear(in_features=64, out_features=16, bias=True)
  )
)


In [7]:
# Блок 6. Обучение seq2seq

def train_seq2seq(model, dataloader, epochs=200, lr=0.005):
    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for src, tgt_inp, tgt_out in dataloader:
            src = src.to(device)
            tgt_inp = tgt_inp.to(device)
            tgt_out = tgt_out.to(device)

            optimizer.zero_grad()
            logits = model(src, tgt_inp)
            loss = criterion(
                logits.view(-1, len(tgt_vocab)),
                tgt_out.view(-1)
            )
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 20 == 0:
            print(f"Epoch {epoch}: loss={total_loss/len(dataloader):.4f}")

train_seq2seq(model, dataloader, epochs=200, lr=0.005)

Epoch 20: loss=0.2643
Epoch 40: loss=0.1961
Epoch 60: loss=0.1034
Epoch 80: loss=0.0242
Epoch 100: loss=0.0069
Epoch 120: loss=0.0041
Epoch 140: loss=0.0030
Epoch 160: loss=0.0023
Epoch 180: loss=0.0018
Epoch 200: loss=0.0015


In [8]:
# Блок 7. Перевод (greedy decoding)

def translate(model, sentence, max_len=MAX_TGT_LEN):
    model.eval()
    src_ids = encode_src(sentence).unsqueeze(0).to(device)
    with torch.inference_mode():
        hidden = model.encoder(src_ids)

        inp = torch.tensor([[TGT_SOS_IDX]], dtype=torch.long, device=device)
        outputs = []
        for _ in range(max_len):
            emb = model.decoder.embed(inp)
            out, hidden = model.decoder.rnn(emb, hidden)
            logits = model.decoder.fc(out[:, -1, :])
            pred_id = logits.argmax(dim=-1)
            token_id = int(pred_id.item())
            if token_id == TGT_EOS_IDX:
                break
            outputs.append(token_id)
            inp = pred_id.unsqueeze(0)

    tokens = [tgt_itos[i] for i in outputs]
    return " ".join(tokens)

for src, tgt in pairs:
    pred = translate(model, src)
    print("SRC :", src)
    print("TGT :", tgt)
    print("PRED:", pred)
    print("-" * 40)

SRC : i like cats
TGT : мне нравятся кошки
PRED: мне нравятся кошки
----------------------------------------
SRC : i like dogs
TGT : мне нравятся собаки
PRED: мне нравятся собаки
----------------------------------------
SRC : you like cats
TGT : тебе нравятся кошки
PRED: тебе нравятся кошки
----------------------------------------
SRC : you like dogs
TGT : тебе нравятся собаки
PRED: тебе нравятся собаки
----------------------------------------
SRC : cats are animals
TGT : кошки это животные
PRED: кошки это животные
----------------------------------------
SRC : dogs are animals
TGT : собаки это животные
PRED: собаки это животные
----------------------------------------
SRC : i have a dog
TGT : у меня есть собака
PRED: у меня есть собака
----------------------------------------
SRC : i have a cat
TGT : у меня есть кошка
PRED: у меня есть кошка
----------------------------------------
SRC : you have a dog
TGT : у тебя есть собака
PRED: у тебя есть собака
---------------------------------

In [9]:
# Блок 8. BLEU-метрика

def make_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def corpus_bleu(references, hypotheses, max_n=4, smooth=True):
    weights = [1.0/max_n] * max_n
    p_ns = []
    for n in range(1, max_n+1):
        num_clip = 0
        denom = 0
        for ref, hyp in zip(references, hypotheses):
            ref_ngrams = Counter(make_ngrams(ref, n))
            hyp_ngrams = Counter(make_ngrams(hyp, n))
            denom += sum(hyp_ngrams.values())
            for ng in hyp_ngrams:
                num_clip += min(hyp_ngrams[ng], ref_ngrams.get(ng, 0))
        if denom == 0:
            p_n = 0.0
        else:
            p_n = num_clip / denom
        if p_n == 0 and smooth:
            p_n = 1e-8
        p_ns.append(p_n)

    ref_len = sum(len(r) for r in references)
    hyp_len = sum(len(h) for h in hypotheses)
    if hyp_len > ref_len:
        bp = 1.0
    else:
        if hyp_len == 0:
            bp = 0.0
        else:
            bp = math.exp(1 - ref_len / hyp_len)

    score = bp * math.exp(sum(w * math.log(p) for w, p in zip(weights, p_ns)))
    return score

refs = []
hyps = []
for src, tgt in pairs:
    pred = translate(model, src)
    refs.append(tokenize_ru(tgt))
    hyps.append(tokenize_ru(pred))

bleu_4 = corpus_bleu(refs, hyps, max_n=4, smooth=True)
bleu_1 = corpus_bleu(refs, hyps, max_n=1, smooth=True)

print("BLEU-1:", bleu_1)
print("BLEU-4:", bleu_4)

BLEU-1: 1.0
BLEU-4: 1.0


## Выводы по задаче 3

- Был сформирован небольшой параллельный корпус из 10 английских предложений и их русских переводов, описывающих простые конструкции («i/you like/have cats/dogs», «cats/dogs are animals» и т.п.).
- На основе этого корпуса реализована и обучена seq2seq‑модель с рекуррентным энкодером и декодером (GRU) без механизма внимания; обучение проводилось в схеме teacher forcing на уровне токенов.
- После обучения модель на всех тестовых примерах порождает переводы, полностью совпадающие с эталонными строками (для каждой пары SRC/TGT предсказанный PRED идентичен TGT).
- Расчёт метрик показывает, что BLEU‑1 и BLEU‑4 равны 1.0, что означает идеальное совпадение n‑грамм гипотезы и эталона на данном корпусе; для выбранного набора фраз модель фактически выучила точное соответствие между английскими и русскими предложениями.
- При расширении корпуса до более разнообразных и длинных предложений от модели можно ожидать снижения BLEU, поэтому для реальных задач перевода потребуется больше данных, регуляризация и, вероятно, более сложные архитектуры (например, внимание или трансформеры).
